# Vectorless RAG — PageIndex ধারণা (No Vector DB, No Chunking)

এই notebook-এ আমরা শিখব **Vectorless RAG** — একটা reasoning-based retrieval পদ্ধতি যেটাতে কোনো vector database লাগে না, কোনো embedding লাগে না, এমনকি chunking-ও লাগে না।

এই আইডিয়াটা এসেছে **[PageIndex](https://pageindex.ai)** থেকে — একটা tool যেটা document-কে vector-এ না ভেঙে, একটা **hierarchical tree (Table of Contents)** বানায় এবং LLM-কে সেই tree-এর উপর reason করতে দেয়।

## 🔑 মূল ধারণা

> **Traditional Vector RAG** → chunk করো → embed করো → cosine similarity → retrieve করো
> **PageIndex / Vectorless RAG** → tree বানাও → LLM tree-এর উপর reasoning করে → সঠিক section retrieve করে

**Vector RAG-এর মূল সমস্যা:**

```
Similarity ≠ Relevance
```

একটা chunk হয়তো "market conditions" নিয়ে কথা বলছে বলে query-এর সাথে বেশি word মিলে যাচ্ছে (উচ্চ similarity score),
কিন্তু আসল উত্তর আছে অন্য একটা chunk-এ — যেটার সাথে শব্দগত মিল কম, কিন্তু semantically সেটাই সঠিক answer।

## 📚 আজকে যা শিখব

| # | Topic |
|---|-------|
| 1 | Vector RAG কেন professional/structured document-এ fail করে |
| 2 | PDF থেকে LLM দিয়ে hierarchical tree index বানানো (কোনো chunking ছাড়াই) |
| 3 | LLM Tree Search — tree structure-এর উপর LLM reasoning |
| 4 | সম্পূর্ণ end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval — domain knowledge prompt-এ inject করা |
| 6 | Vector RAG vs Vectorless RAG — কখন কোনটা ব্যবহার করব |

> এই notebook-এ আমরা **PageIndex-এর official cloud SDK ব্যবহার করছি না** — বরং একই ধারণাটা **`langchain-anthropic` (Claude)** দিয়ে from-scratch তৈরি করছি, যাতে পুরো pipeline-টা local এবং কোনো extra API key (PageIndex/OpenAI) ছাড়াই চলে। Source/reference হিসেবে ব্যবহার করা হয়েছে PageIndex-এর official crash-course notebook (Krish Naik, [@krishnaik06](https://youtube.com/@krishnaik06))।


---
## কেন এটা দরকার? Vector RAG-এর সমস্যা

আগের notebook-এ (`RAG/notebook/text-splitting.ipynb`) আমরা শিখেছি traditional RAG pipeline:

```
Document → chunk করা → embed করা (vector) → FAISS/Chroma-তে রাখা
Query    → embed করা → cosine similarity দিয়ে top-k chunk খোঁজা
```

এটা ছোট, বৈচিত্র্যময় document-এ (FAQ, product description) ভালো কাজ করে। কিন্তু **লম্বা, professionally-structured document**-এ (research paper, legal doc, course syllabus, annual report) সমস্যা হয়:

| সমস্যা | কেন হয় |
|---|---|
| Arbitrary chunking | Section-এর boundary না মেনে fixed-size (যেমন ৫০০ token) কেটে ফেলে — একটা বাক্য দুই chunk-এ ভাগ হয়ে যেতে পারে |
| Similarity ≠ Relevance | শব্দগত মিল (embedding distance) বেশি মানেই answer relevant না |
| কোনো traceability নেই | Retrieved chunk-এ section title/page number থাকে না — কোথা থেকে এলো বোঝা যায় না |
| Domain knowledge inject করা কঠিন | Embedding model fine-tune করা লাগে — সময়সাপেক্ষ ও ব্যয়বহুল |

**Vectorless RAG-এর সমাধান:** document-টাকে ভাঙার বদলে, তার **natural structure (chapter → section → sub-section)** ধরে রাখা হয় একটা tree-তে, এবং retrieval-এর সময় LLM-কে সেই tree পড়ে reasoning করতে দেওয়া হয় — ঠিক যেভাবে একজন মানুষ Table of Contents দেখে বলে দিতে পারে উত্তর কোথায় আছে।


---
## Step 1 — Environment Setup

অন্য সব notebook-এর মতোই `.env` থেকে `ANTHROPIC_API_KEY` load করে `ChatAnthropic` LLM তৈরি করা হচ্ছে। এই পুরো pipeline-এ **শুধু Anthropic API** লাগবে — আলাদা কোনো embedding model বা vector DB লাগবে না।


In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
    max_tokens=8192,
)
print("LLM ready:", llm.model)


G:\all projects\AI agents code\agentic-course-krish-naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ANTHROPIC_API_KEY: True
LLM ready: claude-haiku-4-5-20251001


---
## Step 2 — Sample Document তৈরি করা

Tree structure-এর সুবিধা দেখাতে হলে একটা **multi-page, multi-section** document দরকার — একটা এক-পৃষ্ঠার PDF দিয়ে hierarchy বোঝা যাবে না।

তাই `fpdf2` দিয়ে একটা ছোট **AI Course Syllabus** PDF বানানো হচ্ছে (Krish Naik Academy-এর আসল syllabus থেকে ৫টা module নেওয়া হয়েছে) — কয়েকটা module-এর নিচে একাধিক sub-topic আছে, যেটা ঠিক আমাদের দরকারি hierarchical structure তৈরি করবে।


In [2]:
import os
from fpdf import FPDF

os.makedirs("data", exist_ok=True)
PDF_PATH = "data/ai_course_syllabus.pdf"

# Krish Naik Academy-র "Advanced Route of Learning AI" syllabus থেকে নেওয়া (condensed)
SECTIONS = [
    ("Module 1: Neural Network Refresher", [
        "Backpropagation fundamentals",
        "Activation functions (ReLU, GELU, SiLU)",
        "Optimizers (SGD, Adam, AdamW)",
        "PyTorch basics",
    ]),
    ("Module 2: Hardware", [
        "GPU architecture basics",
        "TPU vs GPU tradeoffs",
        "Apple Silicon for ML workloads",
        "Compute infrastructure planning",
    ]),
    ("Module 9: Modern LLM Finetuning", [
        "The LLM Development Lifecycle",
        "Pre-Training Deep Dive",
        "Data Preparation for Fine-Tuning",
        "Parameter-Efficient Fine-Tuning (PEFT): LoRA, QLoRA, DoRA",
        "Supervised Fine-Tuning (SFT)",
        "Preference Alignment: RLHF, DPO, ORPO",
        "Evaluation and benchmarking",
        "Quantization and deployment prep",
    ]),
    ("Module 17: RAG", [
        "Vanilla RAG and chunking strategies",
        "BM25, SPLADE and multi-vector ColBERT",
        "Hybrid RAG and rerankers",
        "Self RAG, Corrective RAG, Adaptive RAG",
        "Agentic RAG",
        "Vectorless RAG - tree based retrieval without embeddings",
    ]),
    ("Module 20: Agents", [
        "ReAct pattern",
        "MCP - Model Context Protocol",
        "LangGraph state machines",
        "Multi-agent orchestration",
    ]),
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# multi_cell-এর পর default cursor page-এর ডান প্রান্তে থেকে যায় (new_x=XPos.RIGHT),
# তাই প্রতিবার left margin-এ ফেরত আনতে new_x="LMARGIN" দেওয়া হচ্ছে।
def line(text: str) -> None:
    pdf.multi_cell(0, 8, text, new_x="LMARGIN", new_y="NEXT")

# Title page
pdf.add_page()
pdf.set_font("Helvetica", "B", 18)
line("ADVANCED ROUTE OF LEARNING AI")
pdf.set_font("Helvetica", "", 12)
line("Comprehensive Syllabus - Krish Naik Academy | 2025-2026 Cohort")
pdf.ln(5)
pdf.set_font("Helvetica", "B", 13)
line("Table of Contents")
pdf.set_font("Helvetica", "", 11)
for title, _ in SECTIONS:
    line(title)

# প্রতিটা module আলাদা page-এ
for title, topics in SECTIONS:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    line(title)
    pdf.set_font("Helvetica", "", 11)
    for t in topics:
        line(f"- {t}")

pdf.output(PDF_PATH)
print(f"✅ Sample PDF তৈরি হলো: {PDF_PATH}")


✅ Sample PDF তৈরি হলো: data/ai_course_syllabus.pdf


---
## Step 3 — PDF Load করা (page-wise, chunking ছাড়াই)

লক্ষ্য করো — এখানে `RecursiveCharacterTextSplitter` ব্যবহার করা হচ্ছে **না**। প্রতিটা page-কে তার আসল অবস্থাতেই রাখা হচ্ছে, কারণ tree বানানোর সময় LLM নিজেই section boundary বুঝে নেবে।


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"মোট পেজ: {len(pages)}\n")
for p in pages:
    preview = p.page_content[:80].replace("\n", " ")
    print(f"[Page {p.metadata['page'] + 1}] ({len(p.page_content)} chars) — {preview}...")


C:\Users\Nibras\AppData\Local\Temp\ipykernel_21568\909460309.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


মোট পেজ: 6

[Page 1] (229 chars) — ADVANCED ROUTE OF LEARNING AI Comprehensive Syllabus - Krish Naik Academy | 2025...
[Page 2] (156 chars) — Module 1: Neural Network Refresher - Backpropagation fundamentals - Activation f...
[Page 3] (134 chars) — Module 2: Hardware - GPU architecture basics - TPU vs GPU tradeoffs - Apple Sili...
[Page 4] (319 chars) — Module 9: Modern LLM Finetuning - The LLM Development Lifecycle - Pre-Training D...
[Page 5] (233 chars) — Module 17: RAG - Vanilla RAG and chunking strategies - BM25, SPLADE and multi-ve...
[Page 6] (119 chars) — Module 20: Agents - ReAct pattern - MCP - Model Context Protocol - LangGraph sta...


---
## Step 4 — Hierarchical Tree Index বানানো (LLM দিয়ে)

এখানেই **PageIndex-এর মূল আইডিয়া**: পুরো document-এর page text একসাথে LLM-কে দিয়ে বলা হয় — "এখান থেকে একটা Table-of-Contents-এর মতো tree বানাও।"

প্রতিটা node-এ থাকবে:
- `node_id` — unique identifier (retrieval-এর সময় ব্যবহার হবে)
- `title` — section-এর heading
- `page_index` — কোন page-এ আছে
- `summary` — সংক্ষিপ্ত সারাংশ
- `children` — nested sub-section (থাকলে)

আগের notebook-এ (`langchain/5-structured-output.ipynb`) শেখা **`with_structured_output()`** এখানে recursive Pydantic model-এর সাথে ব্যবহার করা হচ্ছে — এতে LLM-এর output সবসময় valid, typed tree structure হবে, manual JSON parsing লাগবে না।


In [4]:
from pydantic import BaseModel, Field


class TreeNode(BaseModel):
    """Document tree-র একটা node — section বা sub-section।"""
    node_id: str = Field(description="Unique 4-digit ID, e.g. '0001'")
    title: str = Field(description="Section-এর heading")
    page_index: int = Field(description="এই section যেই page-এ আছে (1-indexed)")
    summary: str = Field(description="এই section-এর ১-২ বাক্যের সারাংশ")
    children: list["TreeNode"] = Field(
        default_factory=list, description="Nested sub-section গুলো, না থাকলে খালি list"
    )


class DocumentTree(BaseModel):
    """পুরো document-এর top-level section গুলোর list।"""
    nodes: list[TreeNode] = Field(description="Document-এর top-level section গুলো")


TreeNode.model_rebuild()
print("✅ TreeNode / DocumentTree schema রেডি")


✅ TreeNode / DocumentTree schema রেডি


In [5]:
from langchain_core.messages import HumanMessage

TREE_BUILD_PROMPT = """তুমি একজন document structure analyst।
নিচে একটা PDF-এর প্রতিটা page-এর raw text দেওয়া আছে (page number সহ)।

তোমার কাজ: এই document-এর একটা hierarchical tree index বানানো — ঠিক যেমন একটা বইয়ের Table of Contents হয়।
- প্রতিটা Module একটা top-level node হবে
- Module-এর ভেতরের topic গুলো (থাকলে) children node হবে
- প্রতিটা node-কে ইউনিক ৪-digit node_id দাও ("0000", "0001", ...)
- Fixed-size chunking করো না — section-এর প্রাকৃতিক boundary অনুযায়ী ভাগ করো
- Title page / Table of Contents page-কে আলাদা node বানানোর দরকার নেই

Document pages:
{pages_text}
"""


def build_tree_from_pages(pages, llm) -> DocumentTree:
    """PDF-এর সব page একসাথে LLM-কে দিয়ে hierarchical tree বানানো হয় (chunking ছাড়াই)।"""
    pages_text = "\n\n".join(
        f"[PAGE {p.metadata['page'] + 1}]\n{p.page_content}" for p in pages
    )
    structured_llm = llm.with_structured_output(DocumentTree)
    return structured_llm.invoke(
        [HumanMessage(content=TREE_BUILD_PROMPT.format(pages_text=pages_text))]
    )


document_tree = build_tree_from_pages(pages, llm)
print(f"✅ Tree তৈরি হয়েছে — top-level sections: {len(document_tree.nodes)}")


✅ Tree তৈরি হয়েছে — top-level sections: 5


In [6]:
def print_tree(nodes: list[TreeNode], indent: int = 0) -> None:
    """Tree-টা recursively human-readable আকারে print করে।"""
    for n in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        print(f"{prefix}[{n.node_id}] {n.title}  (p.{n.page_index})")
        if n.children:
            print_tree(n.children, indent + 1)


def count_nodes(nodes: list[TreeNode]) -> int:
    """Tree-তে মোট কতগুলো node আছে (nested সহ) গণনা করে।"""
    total = len(nodes)
    for n in nodes:
        total += count_nodes(n.children)
    return total


print("📚 সম্পূর্ণ Document Tree:\n")
print_tree(document_tree.nodes)
print(f"\n🔢 মোট node সংখ্যা: {count_nodes(document_tree.nodes)}")
print("   প্রতিটা node = document-এর একটা retrievable section")


📚 সম্পূর্ণ Document Tree:

[0001] Module 1: Neural Network Refresher  (p.2)
  └─ [0002] Backpropagation fundamentals  (p.2)
  └─ [0003] Activation functions (ReLU, GELU, SiLU)  (p.2)
  └─ [0004] Optimizers (SGD, Adam, AdamW)  (p.2)
  └─ [0005] PyTorch basics  (p.2)
[0006] Module 2: Hardware  (p.3)
  └─ [0007] GPU architecture basics  (p.3)
  └─ [0008] TPU vs GPU tradeoffs  (p.3)
  └─ [0009] Apple Silicon for ML workloads  (p.3)
  └─ [0010] Compute infrastructure planning  (p.3)
[0011] Module 9: Modern LLM Finetuning  (p.4)
  └─ [0012] The LLM Development Lifecycle  (p.4)
  └─ [0013] Pre-Training Deep Dive  (p.4)
  └─ [0014] Data Preparation for Fine-Tuning  (p.4)
  └─ [0015] Parameter-Efficient Fine-Tuning (PEFT): LoRA, QLoRA, DoRA  (p.4)
  └─ [0016] Supervised Fine-Tuning (SFT)  (p.4)
  └─ [0017] Preference Alignment: RLHF, DPO, ORPO  (p.4)
  └─ [0018] Evaluation and benchmarking  (p.4)
  └─ [0019] Quantization and deployment prep  (p.4)
[0020] Module 17: RAG  (p.5)
  └─ [0021] Vanill

---
## Step 5 — LLM Tree Search: Vectorless RAG-এর মূল অংশ

এইখানেই vector RAG থেকে সম্পূর্ণ ভিন্ন approach শুরু হয়:

**Vector RAG retrieval:**
```
query → embed করা → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunk
```
*সমস্যা: কী similar সেটা খুঁজে বের করে, কী relevant সেটা না*

**Vectorless RAG retrieval:**
```
query + tree → LLM reasoning করে → "node 0009 আর 0010-এ উত্তর আছে"
```
*সুবিধা: LLM document-এর structure, context, এবং intent বুঝে বেছে নেয়*

Token বাঁচাতে পুরো tree-এর `text`/`children` না পাঠিয়ে, শুধু `node_id`, `title`, `page`, `summary` compress করে পাঠানো হয়।


In [7]:
import json


class SearchResult(BaseModel):
    """Tree search-এর ফলাফল — selected node ID এবং সংক্ষিপ্ত reasoning।

    node_list আগে রাখা হয়েছে যাতে verbose thinking output token limit-এ
    truncate হয়ে গেলেও আসল দরকারি field (node_list) আগেই তৈরি হয়ে যায়।
    """
    node_list: list[str] = Field(description="যেসব node_id-তে query-এর উত্তর থাকতে পারে")
    thinking: str = Field(description="সংক্ষেপে ১-২ বাক্যে reasoning — কেন এই node গুলো relevant")


def _compress_tree(nodes: list[TreeNode]) -> list[dict]:
    """Token বাঁচাতে শুধু node_id/title/page/summary পাঠানো হয় (full text নয়)।"""
    out = []
    for n in nodes:
        entry = {"node_id": n.node_id, "title": n.title, "page": n.page_index, "summary": n.summary}
        if n.children:
            entry["children"] = _compress_tree(n.children)
        out.append(entry)
    return out


TREE_SEARCH_PROMPT = """তুমি একজন document expert। নিচে একটা query এবং document-এর tree structure
(Table of Contents-এর মতো) দেওয়া আছে।

কাজ: কোন কোন node_id-তে query-এর উত্তর থাকতে পারে সেটা ধাপে ধাপে চিন্তা করে বের করো।

Query: {query}

Document Tree (JSON):
{tree_json}
"""


def llm_tree_search(query: str, tree_nodes: list[TreeNode], llm) -> SearchResult:
    """
    Vector RAG-এর 'embed + cosine similarity'-র বদলে —
    এখানে LLM নিজেই tree পড়ে reasoning করে বলে দেয় উত্তর কোথায় আছে।
    """
    tree_json = json.dumps(_compress_tree(tree_nodes), indent=2, ensure_ascii=False)
    structured_llm = llm.with_structured_output(SearchResult)
    return structured_llm.invoke(
        [HumanMessage(content=TREE_SEARCH_PROMPT.format(query=query, tree_json=tree_json))]
    )


# ── টেস্ট ──
query = "Modern LLM finetuning-এ কী কী পড়ানো হয়?"
result = llm_tree_search(query, document_tree.nodes, llm)

print(f"🔍 Query: {query}\n")
print("🧠 Reasoning:", result.thinking)
print("\n🎯 Selected node IDs:", result.node_list)


🔍 Query: Modern LLM finetuning-এ কী কী পড়ানো হয়?

🧠 Reasoning: Query: "Modern LLM finetuning-এ কী কী পড়ানো হয়?" (What is taught in Modern LLM finetuning?)

এই query-টা সরাসরি Module 9 "Modern LLM Finetuning" এর সাথে ম্যাচ করে। 

পদ্ধতিগত বিশ্লেষণ:

1. **মূল মডিউল (0011)**: "Modern LLM Finetuning" - এটি সরাসরি query-র উত্তর দেবে এবং সমস্ত subtopics কভার করবে।

2. **Sub-topics যা Modern LLM Finetuning-এ পড়ানো হয়**:
   - **0012**: LLM Development Lifecycle - ফাইনটিউনিং-এর সম্পূর্ণ প্রক্রিয়া
   - **0013**: Pre-Training Deep Dive - ফাইনটিউনিং-এর আগের ধাপ
   - **0014**: Data Preparation for Fine-Tuning - ডেটা প্রস্তুতি
   - **0015**: PEFT (LoRA, QLoRA, DoRA) - প্যারামিটার-দক্ষ ফাইনটিউনিং কৌশল
   - **0016**: Supervised Fine-Tuning (SFT) - তত্ত্ববদ্ধ ফাইনটিউনিং পদ্ধতি
   - **0017**: Preference Alignment (RLHF, DPO, ORPO) - মডেল এলাইনমেন্ট কৌশল
   - **0018**: Evaluation and benchmarking - মূল্যায়ন পদ্ধতি
   - **0019**: Quantization and deployment prep - ডিপ্লয়মেন্ট প্রস্তুতি

3. **অন্য

---
## Step 6 — সম্পূর্ণ End-to-End Vectorless RAG Pipeline

তিনটা ধাপ:
1. **Tree Search** → LLM relevant `node_id` বেছে নেয়
2. **Retrieve** → সেই node গুলোর actual content বের করা হয়
3. **Generate** → LLM section title + page number cite করে একটা grounded answer লেখে

**এটা vector RAG থেকে ভালো কেন:**
- Retrieved content-এর সাথে title + page number থাকে (traceable)
- LLM ঠিক কোন section থেকে উত্তর এলো সেটা cite করতে পারে
- Irrelevant chunk থেকে hallucination হওয়ার সম্ভাবনা কমে যায়


In [8]:
def find_nodes_by_ids(nodes: list[TreeNode], target_ids: list[str]) -> list[TreeNode]:
    """Tree recursively walk করে target_ids-এর সাথে মিলে যাওয়া node গুলো বের করে।"""
    found = []
    for n in nodes:
        if n.node_id in target_ids:
            found.append(n)
        found.extend(find_nodes_by_ids(n.children, target_ids))
    return found


ANSWER_PROMPT = """তুমি একজন expert document analyst।
নিচের context ব্যবহার করে প্রশ্নের উত্তর দাও। প্রতিটা তথ্যের পাশে section title এবং page number cite করো।
Context-এ যা নেই সেটা নিজে থেকে বানিয়ে বলবে না। উত্তর বাংলায়, সংক্ষেপে দাও।

Question: {question}

Context:
{context}
"""


def generate_answer(query: str, nodes: list[TreeNode], llm) -> str:
    """Retrieved node গুলো থেকে context বানিয়ে page-cited answer generate করে।"""
    if not nodes:
        return "⚠️ এই বিষয়ে document-এ কোনো relevant section পাওয়া যায়নি।"

    context = "\n\n---\n\n".join(
        f"[Section: '{n.title}' | Page {n.page_index}]\n{n.summary}" for n in nodes
    )
    response = llm.invoke([HumanMessage(content=ANSWER_PROMPT.format(question=query, context=context))])
    return response.content


def vectorless_rag(query: str, tree_nodes: list[TreeNode], llm, verbose: bool = True) -> str:
    """
    সম্পূর্ণ Vectorless RAG pipeline:
    Step 1: llm_tree_search   → relevant node_id বের করা
    Step 2: find_nodes_by_ids → node-এর content retrieve করা
    Step 3: generate_answer   → cited, grounded answer বানানো
    """
    search_result = llm_tree_search(query, tree_nodes, llm)
    nodes = find_nodes_by_ids(tree_nodes, search_result.node_list)

    if verbose:
        print(f"🧠 Reasoning: {search_result.thinking[:200]}...")
        print(f"📄 Retrieved sections: {[n.title for n in nodes]}\n")

    return generate_answer(query, nodes, llm)


# ── পুরো pipeline একবার চালানো ──
answer = vectorless_rag(query="RAG module-এ কোন কোন topic পড়ানো হয়?", tree_nodes=document_tree.nodes, llm=llm)
print(f"📝 উত্তর:\n{answer}")


🧠 Reasoning: Query-এ "RAG module-এ কোন কোন topic পড়ানো হয়" - এটা RAG module-এর সকল topics খুঁজছে।

Document tree-তে "Module 17: RAG" হল node_id "0020" যার সারণী হলো Comprehensive coverage of Retrieval-Augmented ...
📄 Retrieved sections: ['Module 17: RAG', 'Vanilla RAG and chunking strategies', 'BM25, SPLADE and multi-vector ColBERT', 'Hybrid RAG and rerankers', 'Self RAG, Corrective RAG, Adaptive RAG', 'Agentic RAG', 'Vectorless RAG - tree based retrieval without embeddings']



📝 উত্তর:
# RAG Module-এ পড়ানো বিষয়সমূহ

RAG Module 17-এ (Page 5) নিচের topic গুলি কভার করা হয়:

1. **Vanilla RAG এবং Chunking Strategies** - মৌলিক RAG ধারণা এবং ডকুমেন্ট রিট্রিভালের জন্য টেক্সট চাংকিং পদ্ধতি

2. **BM25, SPLADE এবং Multi-vector ColBERT** - রিট্রিভাল এলগরিদম (Sparse এবং Dense রিট্রিভাল)

3. **Hybrid RAG এবং Rerankers** - একাধিক রিট্রিভাল পদ্ধতি সমন্বয় এবং রেরাংকিং

4. **Self RAG, Corrective RAG, Adaptive RAG** - স্ব-সংশোধন এবং খাপ খাইয়ে নেওয়ার মেকানিজম সহ উন্নত RAG ভেরিয়েন্ট

5. **Agentic RAG** - এজেন্ট-ভিত্তিক সিস্টেমের সাথে সংযোজন

6. **Vectorless RAG** - এমবেডিং ছাড়াই ট্রি-ভিত্তিক রিট্রিভাল পদ্ধতি


In [9]:
# ── একাধিক query দিয়ে টেস্ট ──
test_queries = [
    "Modern LLM finetuning-এ কী কী পড়ানো হয়?",
    "Neural Network Refresher module-এ কী শেখানো হয়?",
    "Agent-সংক্রান্ত কোন কোন বিষয় আছে?",
]

for q in test_queries:
    print("=" * 60)
    print(f"প্রশ্ন: {q}")
    ans = vectorless_rag(q, document_tree.nodes, llm, verbose=False)
    print(f"উত্তর:\n{ans}")
print("=" * 60)


প্রশ্ন: Modern LLM finetuning-এ কী কী পড়ানো হয়?


উত্তর:
# Modern LLM Finetuning-এ পড়ানো বিষয়সমূহ

Module 9: Modern LLM Finetuning (Page 4)-তে নিম্নলিখিত বিষয়গুলি অন্তর্ভুক্ত রয়েছে:

1. **Data Preparation for Fine-Tuning** - ডেটা সংগ্রহ, পরিষ্কারকরণ এবং প্রস্তুতি কৌশল

2. **Supervised Fine-Tuning (SFT)** - তত্ত্বাবধানে ভাষা মডেল ফাইনটিউনিং পদ্ধতি

3. **Parameter-Efficient Fine-Tuning (PEFT)** - ন্যূনতম পরামিটার আপডেট সহ ফাইনটিউনিং কৌশল (LoRA, QLoRA, DoRA)

4. **Preference Alignment** - মানব পছন্দের সাথে মডেল আউটপুট সংযুক্তির কৌশল (RLHF, DPO, ORPO)

5. **Evaluation and Benchmarking** - ফাইনটিউন করা মডেল মূল্যায়নের পদ্ধতি

6. **Quantization and Deployment Prep** - মডেল স্থাপনার জন্য কোয়ান্টাইজেশন কৌশল
প্রশ্ন: Neural Network Refresher module-এ কী শেখানো হয়?


উত্তর:
# Neural Network Refresher Module-এ শেখানো বিষয়গুলো

**Module 1: Neural Network Refresher** (Page 2)-এ নিম্নলিখিত বিষয়গুলো কভার করা হয়:

1. **Backpropagation fundamentals** - নিউরাল নেটওয়ার্কের মাধ্যমে গ্র্যাডিয়েন্ট কীভাবে পিছনের দিকে প্রবাহিত হয় তা বোঝা (Page 2)

2. **Activation functions (ReLU, GELU, SiLU)** - বিভিন্ন ধরনের activation function এবং তাদের ব্যবহার (Page 2)

3. **Optimizers (SGD, Adam, AdamW)** - নিউরাল নেটওয়ার্ক প্রশিক্ষণের জন্য সাধারণ অপ্টিমাইজেশন অ্যালগরিদম (Page 2)

4. **PyTorch basics** - নিউরাল নেটওয়ার্ক বাস্তবায়নের জন্য PyTorch ফ্রেমওয়ার্কের পরিচয় (Page 2)
প্রশ্ন: Agent-সংক্রান্ত কোন কোন বিষয় আছে?


উত্তর:
# Agent-সংক্রান্ত বিষয়সমূহ:

1. **Agentic RAG** (পৃষ্ঠা ৫) - RAG এবং agent-ভিত্তিক সিস্টেমের সংযোগ

2. **AI Agents ডিজাইন ও বাস্তবায়ন** (পৃষ্ঠা ৬) - reasoning pattern, protocol, state management এবং multi-agent system

3. **ReAct Pattern** (পৃষ্ঠা ৬) - স্বচ্ছ সিদ্ধান্ত গ্রহণের জন্য Reasoning এবং Acting pattern

4. **MCP - Model Context Protocol** (পৃষ্ঠা ৬) - AI মডেলে context এবং communication পরিচালনার মান প্রোটোকল

5. **LangGraph State Machines** (পৃষ্ঠা ৬) - Agent workflow পরিচালনার জন্য state machine বাস্তবায়ন

6. **Multi-agent Orchestration** (পৃষ্ঠা ৬) - একাধিক স্বায়ত্তশাসিত agent-এর মধ্যে সমন্বয় এবং পারস্পরিক ক্রিয়া পরিচালনা


---
## Step 7 — Expert-Guided Retrieval (Domain Knowledge Injection)

Vector RAG-এ domain expertise inject করতে হলে **embedding model fine-tune** করা লাগে — সময়সাপেক্ষ ও ব্যয়বহুল।

Vectorless RAG-এ এটা শুধু **prompt-এ কিছু rule যোগ করা**:

```
"EBITDA-সংক্রান্ত query হলে → MD&A section দেখো"
"Risk-সংক্রান্ত query হলে   → Item 1A দেখো"
```

এতে finance, legal, medical — যেকোনো domain-এ কোনো model training ছাড়াই instantly adapt করা যায়। আমাদের example-এ, একজন senior instructor-এর মতো module-routing rule ব্যবহার করছি।


In [10]:
COURSE_EXPERT_RULES = """
Module routing rules (senior instructor-এর domain knowledge):
- Neural network internals, backprop, optimizer প্রশ্ন → Module 1
- GPU/TPU/hardware/compute infra প্রশ্ন            → Module 2
- Fine-tuning, LoRA, PEFT, RLHF, DPO প্রশ্ন          → Module 9
- Retrieval, chunking, RAG variants প্রশ্ন           → Module 17
- Agent, MCP, LangGraph, multi-agent প্রশ্ন           → Module 20
- "production / deployment" প্রশ্ন                   → Module 9 (quantization) + Module 20 (agents)
"""


def llm_tree_search_with_expert(
    query: str, tree_nodes: list[TreeNode], expert_rules: str, llm
) -> SearchResult:
    """llm_tree_search()-এর মতোই, কিন্তু prompt-এ domain expert rule inject করা হয়।"""
    tree_json = json.dumps(_compress_tree(tree_nodes), indent=2, ensure_ascii=False)
    prompt = f"""তুমি একজন domain expert। নিচের expert routing rule অনুসরণ করে বলো কোন node_id-তে উত্তর আছে।

Query: {query}

Document Tree (JSON):
{tree_json}

Expert Routing Rules:
{expert_rules}
"""
    structured_llm = llm.with_structured_output(SearchResult)
    return structured_llm.invoke([HumanMessage(content=prompt)])


def expert_rag(query: str, tree_nodes: list[TreeNode], rules: str, llm) -> str:
    """Domain expert rule দিয়ে guided vectorless RAG।"""
    result = llm_tree_search_with_expert(query, tree_nodes, rules, llm)
    nodes = find_nodes_by_ids(tree_nodes, result.node_list)
    return generate_answer(query, nodes, llm)


# ── Expert rules ছাড়া vs সহ তুলনা ──
query = "Production deployment-এর জন্য কী কী লাগবে?"

basic = llm_tree_search(query, document_tree.nodes, llm)
print("── Expert rules ছাড়া ──")
print("Nodes:", basic.node_list)

print()

guided = llm_tree_search_with_expert(query, document_tree.nodes, COURSE_EXPERT_RULES, llm)
print("── Expert rules সহ ──")
print("Nodes:", guided.node_list)
print("Reasoning:", guided.thinking[:300])


── Expert rules ছাড়া ──
Nodes: ['0019', '0012', '0010', '0006', '0020']



── Expert rules সহ ──
Nodes: ['0019', '0027', '0030', '0031']
Reasoning: User is asking about "Production deployment-এর জন্য কী কী লাগবে?" which translates to "What do I need for production deployment?"

According to the expert routing rules provided:
- "production / deployment" questions → Module 9 (quantization) + Module 20 (agents)

In Module 9 (Modern LLM Finetuning)


---
## Step 8 — Vector RAG vs Vectorless RAG: পাশাপাশি তুলনা

| বিষয় | Traditional Vector RAG | Vectorless RAG (PageIndex-style) |
|---|---|---|
| Document প্রস্তুতি | Fixed-size chunk-এ ভাগ করা | Hierarchical tree বানানো |
| Indexing | প্রতিটা chunk embed করা | LLM structure পড়ে |
| Storage | Vector database (FAISS/Chroma) | সাধারণ JSON |
| Query processing | Query embed → ANN search | LLM tree-এর উপর reasoning করে |
| যা retrieve হয় | Flat, anonymous chunk | Named section + page reference |
| Explainability | ❌ Opaque similarity score | ✅ Traceable reasoning + citation |
| Domain expertise | ❌ Embedding fine-tune লাগে | ✅ শুধু prompt-এ rule যোগ করলেই হয় |
| Infrastructure | Pinecone/FAISS/ChromaDB লাগে | কোনো vector DB লাগে না |
| সবচেয়ে উপযোগী | ছোট, বৈচিত্র্যময় document (FAQ) | লম্বা, structured document (report, syllabus, legal doc) |
| FinanceBench accuracy (PageIndex benchmark) | ~80% | **~98.7%** |

### কখন কোনটা ব্যবহার করব

**Vector RAG ব্যবহার করো যখন:**
- Document ছোট এবং বৈচিত্র্যময় (FAQ, product description)
- Semantic paraphrase matching গুরুত্বপূর্ণ
- লক্ষ লক্ষ document-এ sub-second retrieval দরকার

**Vectorless RAG ব্যবহার করো যখন:**
- Document লম্বা এবং professionally structured (report, manual, legal doc, syllabus)
- Traceable, cited answer দরকার
- Domain expertise দিয়ে retrieval guide করা দরকার
- Vector DB infrastructure এড়াতে চাও


---
## Step 9 — Production-এ ব্যবহারের বিকল্প

এই notebook-এ আমরা concept-টা **from scratch, শুধু Anthropic দিয়ে** implement করেছি — শেখার জন্য এটাই সবচেয়ে ভালো উপায়, কারণ ভেতরে কী হচ্ছে পুরোটা দেখা যায়।

Production-এ আসল **PageIndex** ব্যবহার করতে চাইলে এই বিকল্পগুলো আছে:

| বিকল্প | কখন ব্যবহার করবে |
|---|---|
| **PageIndex Cloud SDK** ([dash.pageindex.ai](https://dash.pageindex.ai/api-keys)) | Tree building/hosting নিজে manage করতে না চাইলে |
| **PageIndex Chat API** | LLM call নিজে manage না করে সরাসরি doc-এর সাথে chat করতে চাইলে |
| **Self-hosted open source** ([github.com/VectifyAI/PageIndex](https://github.com/VectifyAI/PageIndex)) | পুরো data privately/on-prem রাখতে চাইলে |

> Source: Krish Naik-এর *PageIndex — Vectorless RAG Crash Course* notebook ([@krishnaik06](https://youtube.com/@krishnaik06))।


---
## ✅ Summary — কী শিখলাম

এই notebook-এ আমরা সম্পূর্ণ একটা **Vectorless RAG** system বানিয়েছি:

1. **`build_tree_from_pages()`** — PDF থেকে chunking ছাড়াই hierarchical tree বানানো (`with_structured_output` + recursive Pydantic model)
2. **`llm_tree_search()`** — LLM tree-র উপর reasoning করে relevant node খুঁজে বের করা
3. **`find_nodes_by_ids()`** — Tree থেকে actual section content retrieve করা
4. **`generate_answer()`** — Cited, grounded answer তৈরি করা
5. **`vectorless_rag()`** — উপরের সব একসাথে জোড়া লাগানো full pipeline
6. **`expert_rag()`** — কোনো model fine-tuning ছাড়াই domain rule দিয়ে retrieval guide করা

### মূল takeaway

- **`Similarity ≠ Relevance`** — vector search-এর মূল দুর্বলতা
- Tree-based reasoning দেয় **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection মানে শুধু **prompt engineering** — কোনো model training লাগে না
- ছোট, non-structured document-এ vector RAG ভালো; লম্বা, structured document-এ vectorless RAG ভালো

### 🔗 Reference

- Source notebook: *PageIndex — Vectorless RAG Crash Course* (Krish Naik, [@krishnaik06](https://youtube.com/@krishnaik06))
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
